# RiskBands vs OptimalBinning puro
## comparação metodológica, temporal e visual

Este notebook foi reestruturado para responder de forma explícita à pergunta:

> **Em quais cenários o RiskBands entrega vantagem real sobre o OptimalBinning puro?**

A comparação foi desenhada para ser **justa**:

- mesmo dataset
- mesma variável focal
- mesmo target
- mesmo split temporal
- mesmo limite de número de bins
- mesma leitura OOT

As três abordagens comparadas são:

1. **OptimalBinning puro**
2. **RiskBands sem Optuna**
3. **RiskBands com Optuna**

---

## O que este notebook mede

A análise não fica restrita a IV ou separação estática. O foco está em critérios relevantes para risco de crédito:

- estabilidade temporal por safra
- inversões de event rate entre vintages
- robustez out-of-time
- bins esparsos
- sensibilidade a drift temporal
- penalização de bins instáveis
- score temporal consolidado
- custo computacional adicional

> **Observação importante**  
> O `score temporal consolidado` deste notebook é uma métrica de comparação construída para a demo.  
> Ele **não substitui** métricas oficiais do pacote, mas ajuda a explicar visualmente o trade-off entre separação, estabilidade e custo.


## 1. Setup

Este notebook usa:

- `plotly.graph_objects` para todas as visualizações
- `optbinning` para o benchmark puro
- `riskbands` para os dois cenários temporais
- `sklearn` para medir robustez preditiva com uma regressão logística univariada sobre a variável transformada


In [1]:
import warnings
warnings.filterwarnings("ignore")

import time
import json
import math
from typing import Any

import numpy as np
import pandas as pd

import plotly.graph_objects as go
from plotly.subplots import make_subplots

from IPython.display import display
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

try:
    import riskbands
    from riskbands import Binner
except Exception as e:
    raise ImportError(
        "Não foi possível importar `riskbands`. "
        "Ative o ambiente do projeto antes de executar este notebook."
    ) from e

try:
    from optbinning import OptimalBinning
except Exception as e:
    raise ImportError(
        "Não foi possível importar `optbinning`. "
        "Instale ou ative o ambiente que contém o benchmark puro."
    ) from e

np.random.seed(42)

print("riskbands version:", getattr(riskbands, "__version__", "unknown"))

riskbands version: 1.0.0


## 2. Protocolo de comparação justa

A variável focal será **`utilizacao`**, porque ela foi construída para sofrer:

- drift gradual ao longo das safras
- mudança de regime em meses mais recentes
- aumento de sensibilidade ao default no período final

Isso ajuda a tornar visível a diferença entre:

- um binning puramente estático
- um binning que considera estabilidade temporal
- um binning temporal com busca adicional de hiperparâmetros


In [2]:
FEATURE = "utilizacao"
TARGET = "inadimplente"
TIME_COL = "safra_str"

COMMON_CONFIG = {
    "max_bins": 6,
    "min_event_rate_diff": 0.02,
    "monotonic": None,
}

SPARSE_SHARE_THRESHOLD = 0.05
TRAIN_RATIO = 0.70

print("Feature focal:", FEATURE)
print("Target:", TARGET)
print("Time column:", TIME_COL)
print("Configuração comum:", COMMON_CONFIG)

Feature focal: utilizacao
Target: inadimplente
Time column: safra_str
Configuração comum: {'max_bins': 6, 'min_event_rate_diff': 0.02, 'monotonic': None}


## 3. Geração de base sintética com drift e mudança de regime

A base abaixo foi desenhada para simular um cenário plausível de crédito:

- distribuição de utilização muda ao longo do tempo
- o mix de clientes muda por safra
- o efeito da utilização sobre inadimplência fica mais forte no regime final
- a região OOT é justamente a parte mais desafiadora do período


In [3]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))


def generate_temporal_credit_data(
    n_months: int = 24,
    rows_per_month: int = 500,
    regime_start_month: int = 16,
    random_state: int = 42,
) -> pd.DataFrame:
    rng = np.random.default_rng(random_state)
    months = pd.date_range("2023-01-01", periods=n_months, freq="MS")

    frames = []

    for i, month in enumerate(months, start=1):
        n = rows_per_month
        drift = (i - 1) / max(n_months - 1, 1)
        regime_flag = int(i >= regime_start_month)

        idade = rng.normal(loc=35.5 + 2.0 * drift, scale=8.5, size=n).clip(18, 75)
        renda = rng.lognormal(mean=8.05 + 0.10 * drift, sigma=0.48, size=n)
        utilizacao = rng.beta(
            a=2.0 + 1.8 * drift + 0.9 * regime_flag,
            b=max(5.4 - 1.6 * drift - 0.7 * regime_flag, 1.2),
            size=n
        ).clip(0, 1)

        p_varejo = max(0.42 - 0.08 * drift, 0.16)
        p_autonomo = min(0.19 + 0.04 * regime_flag, 0.28)
        p_premium = max(0.16 - 0.05 * drift, 0.07)
        p_digital = 1.0 - (p_varejo + p_autonomo + p_premium)
        p_digital = max(p_digital, 0.05)

        probs = np.array([p_varejo, p_autonomo, p_premium, p_digital], dtype=float)
        probs = probs / probs.sum()

        segmento = rng.choice(
            ["varejo", "autonomo", "premium", "digital"],
            size=n,
            p=probs
        )

        score_comportamental = (
            655
            + 23 * rng.normal(size=n)
            - 115 * utilizacao
            + 11 * drift
            - 8 * regime_flag
        )

        seg_effect = np.select(
            [
                segmento == "varejo",
                segmento == "autonomo",
                segmento == "premium",
                segmento == "digital",
            ],
            [0.18, 0.42, -0.30, 0.08],
            default=0.0
        )

        regime_interaction = regime_flag * np.where(utilizacao >= 0.55, 0.55, 0.10)

        logit = (
            -2.35
            + 1.05 * utilizacao
            - 0.000055 * renda
            - 0.016 * (idade - 35)
            - 0.0042 * (score_comportamental - 650)
            + seg_effect
            + 0.30 * drift
            + 0.22 * regime_flag
            + regime_interaction
            + rng.normal(0, 0.24, size=n)
        )

        pd_default = sigmoid(logit)
        inadimplente = rng.binomial(1, pd_default, size=n)

        frames.append(
            pd.DataFrame(
                {
                    "safra": month,
                    "idade": idade.round(0),
                    "renda": renda.round(2),
                    "utilizacao": utilizacao.round(4),
                    "segmento": segmento,
                    "score_comportamental": score_comportamental.round(2),
                    "inadimplente": inadimplente,
                    "pd_teorica": pd_default.round(6),
                    "regime_flag": regime_flag,
                }
            )
        )

    df = pd.concat(frames, ignore_index=True)
    df["safra_str"] = df["safra"].dt.strftime("%Y-%m")
    return df


df = generate_temporal_credit_data()
df.head()

,safra,idade,renda,utilizacao,segmento,score_comportamental,inadimplente,pd_teorica,regime_flag,safra_str
0,2023-01-01,38.0,6030.91,0.2753,varejo,627.06,1,0.139119,0,2023-01
1,2023-01-01,27.0,4815.95,0.1989,autonomo,670.04,0,0.118096,0,2023-01
2,2023-01-01,42.0,2218.64,0.5073,autonomo,592.50,0,0.187816,0,2023-01
3,2023-01-01,43.0,1523.55,0.2645,digital,634.88,0,0.115497,0,2023-01
4,2023-01-01,19.0,755.23,0.2853,digital,634.74,0,0.131056,0,2023-01


## 4. Visão geral da base e do choque temporal

In [4]:
overview = (
    df.groupby("safra_str")
    .agg(
        obs=(TARGET, "size"),
        bad_rate=(TARGET, "mean"),
        utilizacao_media=(FEATURE, "mean"),
        regime_flag=("regime_flag", "max"),
    )
    .reset_index()
)

display(df.head())
display(overview)

fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    subplot_titles=(
        "Bad rate por safra",
        "Utilização média por safra"
    ),
    vertical_spacing=0.14
)

fig.add_trace(
    go.Scatter(
        x=overview["safra_str"],
        y=overview["bad_rate"],
        mode="lines+markers",
        name="bad_rate"
    ),
    row=1,
    col=1
)

fig.add_trace(
    go.Scatter(
        x=overview["safra_str"],
        y=overview["utilizacao_media"],
        mode="lines+markers",
        name="utilizacao_media"
    ),
    row=2,
    col=1
)

regime_start = overview.loc[overview["regime_flag"].eq(1), "safra_str"].min()
if pd.notna(regime_start):
    fig.add_vrect(
        x0=regime_start,
        x1=overview["safra_str"].iloc[-1],
        fillcolor="lightgray",
        opacity=0.18,
        line_width=0,
        row="all",
        col=1
    )

fig.update_layout(
    template="plotly_white",
    height=700,
    title="Base sintética com drift e mudança de regime"
)

fig.update_yaxes(title_text="Bad rate", row=1, col=1)
fig.update_yaxes(title_text="Utilização média", row=2, col=1)
fig.update_xaxes(title_text="Safra", row=2, col=1)

fig.show()

,safra,idade,renda,utilizacao,segmento,score_comportamental,inadimplente,pd_teorica,regime_flag,safra_str
0,2023-01-01,38.0,6030.91,0.2753,varejo,627.06,1,0.139119,0,2023-01
1,2023-01-01,27.0,4815.95,0.1989,autonomo,670.04,0,0.118096,0,2023-01
2,2023-01-01,42.0,2218.64,0.5073,autonomo,592.50,0,0.187816,0,2023-01
3,2023-01-01,43.0,1523.55,0.2645,digital,634.88,0,0.115497,0,2023-01
4,2023-01-01,19.0,755.23,0.2853,digital,634.74,0,0.131056,0,2023-01


,safra_str,obs,bad_rate,utilizacao_media,regime_flag
0,2023-01,500,0.100,0.278565,0
1,2023-02,500,0.122,0.283610,0
2,2023-03,500,0.120,0.284615,0
3,2023-04,500,0.148,0.301315,0
4,2023-05,500,0.144,0.318204,0
5,2023-06,500,0.168,0.318847,0
6,2023-07,500,0.150,0.329753,0
7,2023-08,500,0.126,0.340660,0
8,2023-09,500,0.118,0.351345,0
9,2023-10,500,0.128,0.367227,0


## 5. Split temporal treino vs OOT

A divisão temporal é fixa para todos os métodos.  
A parte final da série fica como **OOT**, o que força a comparação em um cenário realmente mais difícil.


In [5]:
ordered_months = sorted(df["safra"].unique())
cut_idx = int(len(ordered_months) * TRAIN_RATIO)

train_months = ordered_months[:cut_idx]
oot_months = ordered_months[cut_idx:]

train_df = df[df["safra"].isin(train_months)].copy()
oot_df = df[df["safra"].isin(oot_months)].copy()

print("Train shape:", train_df.shape)
print("OOT shape:", oot_df.shape)
print("Train bad rate:", round(train_df[TARGET].mean(), 4))
print("OOT bad rate:", round(oot_df[TARGET].mean(), 4))
print("Primeira safra OOT:", oot_df[TIME_COL].min())

Train shape: (8000, 10)
OOT shape: (4000, 10)
Train bad rate: 0.1482
OOT bad rate: 0.3242
Primeira safra OOT: 2024-05


## 6. Funções auxiliares

As funções abaixo cuidam de:

- execução das três abordagens
- padronização das saídas
- construção das métricas temporais
- geração das estruturas usadas nos gráficos


In [6]:
def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out.columns = [
        str(c).strip().lower().replace(" ", "_").replace("(%)", "pct").replace("/", "_")
        for c in out.columns
    ]
    return out


def psi_from_series(train_series: pd.Series, oot_series: pd.Series, eps: float = 1e-6):
    train_dist = train_series.astype(str).value_counts(normalize=True)
    oot_dist = oot_series.astype(str).value_counts(normalize=True)

    psi_table = pd.concat([train_dist, oot_dist], axis=1).fillna(0)
    psi_table.columns = ["train_pct", "oot_pct"]

    psi_table["train_pct"] = psi_table["train_pct"].clip(lower=eps)
    psi_table["oot_pct"] = psi_table["oot_pct"].clip(lower=eps)

    psi_table["psi_component"] = (
        (psi_table["oot_pct"] - psi_table["train_pct"])
        * np.log(psi_table["oot_pct"] / psi_table["train_pct"])
    )

    psi_table = psi_table.reset_index().rename(columns={psi_table.columns[0]: "bin"})
    psi_value = float(psi_table["psi_component"].sum())
    return psi_value, psi_table


def summarize_bins(
    eval_df: pd.DataFrame,
    *,
    bin_col: str,
    target_col: str,
    woe_col: str,
) -> pd.DataFrame:
    grp = (
        eval_df.groupby(bin_col, dropna=False)
        .agg(
            count=(target_col, "size"),
            bad_rate=(target_col, "mean"),
            mean_woe=(woe_col, "mean"),
        )
        .reset_index()
        .sort_values("bad_rate")
        .reset_index(drop=True)
    )
    grp["share"] = grp["count"] / grp["count"].sum()
    grp["sparse_bin"] = grp["share"] < SPARSE_SHARE_THRESHOLD
    return grp


def build_bin_time_table(
    eval_df: pd.DataFrame,
    *,
    time_col: str,
    bin_col: str,
    target_col: str,
) -> pd.DataFrame:
    return (
        eval_df.groupby([time_col, bin_col], dropna=False)
        .agg(
            count=(target_col, "size"),
            bad_rate=(target_col, "mean"),
        )
        .reset_index()
    )


def compute_inversion_ratio(
    time_table: pd.DataFrame,
    *,
    ordered_bins: list[str],
    time_col: str,
    bin_col: str,
) -> float:
    if len(ordered_bins) <= 1:
        return 0.0

    pivot = (
        time_table.pivot(index=time_col, columns=bin_col, values="bad_rate")
        .reindex(columns=ordered_bins)
    )

    total_inv = 0
    total_pairs = 0

    for _, row in pivot.iterrows():
        vals = row.to_numpy(dtype=float)
        for i in range(len(vals) - 1):
            left = vals[i]
            right = vals[i + 1]
            if np.isnan(left) or np.isnan(right):
                continue
            total_pairs += 1
            if right < left:
                total_inv += 1

    return float(total_inv / total_pairs) if total_pairs else 0.0


def compute_event_rate_volatility(
    time_table: pd.DataFrame,
    *,
    ordered_bins: list[str],
    time_col: str,
    bin_col: str,
) -> float:
    pivot = (
        time_table.pivot(index=time_col, columns=bin_col, values="bad_rate")
        .reindex(columns=ordered_bins)
    )
    std_per_bin = pivot.std(skipna=True)
    if std_per_bin.empty:
        return 0.0
    return float(std_per_bin.mean())


def compute_univariate_auc(
    train_eval: pd.DataFrame,
    oot_eval: pd.DataFrame,
    *,
    woe_col: str,
    target_col: str,
) -> tuple[float, float]:
    model = LogisticRegression(max_iter=500)
    model.fit(train_eval[[woe_col]], train_eval[target_col])

    train_pred = model.predict_proba(train_eval[[woe_col]])[:, 1]
    oot_pred = model.predict_proba(oot_eval[[woe_col]])[:, 1]

    train_auc = roc_auc_score(train_eval[target_col], train_pred)
    oot_auc = roc_auc_score(oot_eval[target_col], oot_pred)
    return float(train_auc), float(oot_auc)


def temporal_score_from_metrics(
    *,
    inversion_ratio: float,
    event_rate_volatility: float,
    sparse_bin_ratio: float,
    psi_value: float,
    auc_drop: float,
) -> tuple[float, float]:
    norm_inv = min(max(inversion_ratio, 0.0), 1.0)
    norm_vol = min(max(event_rate_volatility / 0.08, 0.0), 1.0)
    norm_sparse = min(max(sparse_bin_ratio, 0.0), 1.0)
    norm_psi = min(max(psi_value / 0.25, 0.0), 1.0)
    norm_auc_drop = min(max(max(auc_drop, 0.0) / 0.08, 0.0), 1.0)

    penalty = 100 * (
        0.30 * norm_inv
        + 0.20 * norm_vol
        + 0.15 * norm_sparse
        + 0.20 * norm_psi
        + 0.15 * norm_auc_drop
    )

    score = 100 - penalty
    return float(score), float(penalty)


def first_available_attr(obj: Any, names: list[str]):
    for name in names:
        if hasattr(obj, name):
            value = getattr(obj, name)
            return value() if callable(value) else value
    return None


def extract_riskbands_feature_summary(binner: Any, feature: str) -> pd.DataFrame:
    summary_obj = first_available_attr(
        binner,
        ["bin_summary", "summary", "get_summary", "report", "get_report"]
    )

    if summary_obj is None or not isinstance(summary_obj, pd.DataFrame):
        return pd.DataFrame(columns=["variable", "bin", "event_rate"])

    summary = normalize_columns(summary_obj)

    feature_col = None
    for candidate in ["variable", "feature", "name"]:
        if candidate in summary.columns:
            feature_col = candidate
            break

    if feature_col is None or "bin" not in summary.columns:
        return pd.DataFrame(columns=["variable", "bin", "event_rate"])

    summary = summary[summary[feature_col].astype(str) == str(feature)].copy()

    if "event_rate" not in summary.columns:
        if {"event", "count"}.issubset(summary.columns):
            summary["event_rate"] = summary["event"] / summary["count"]
        elif {"event", "non_event"}.issubset(summary.columns):
            summary["event_rate"] = summary["event"] / (summary["event"] + summary["non_event"])
        else:
            summary["event_rate"] = np.nan

    return summary[[feature_col, "bin", "event_rate"]].rename(columns={feature_col: "variable"})


def rounded_key(x: Any, ndigits: int = 10):
    if pd.isna(x):
        return None
    try:
        return round(float(x), ndigits)
    except Exception:
        return str(x)


def build_riskbands_bin_mapping(
    transformed_series: pd.Series,
    feature_summary: pd.DataFrame,
) -> dict[Any, str]:
    unique_woe = sorted(
        [x for x in transformed_series.dropna().unique()],
        key=lambda v: float(v),
        reverse=True
    )

    if feature_summary.empty:
        return {rounded_key(v): f"bin_{i:02d}" for i, v in enumerate(unique_woe, start=1)}

    ordered_bins = (
        feature_summary.sort_values("event_rate", ascending=True)["bin"]
        .astype(str)
        .tolist()
    )

    mapping = {}
    for woe_value, bin_label in zip(unique_woe, ordered_bins):
        mapping[rounded_key(woe_value)] = str(bin_label)

    for i, extra_value in enumerate(unique_woe[len(ordered_bins):], start=len(ordered_bins) + 1):
        mapping[rounded_key(extra_value)] = f"bin_{i:02d}"

    return mapping


def looks_like_bin_labels(series: pd.Series) -> bool:
    vals = series.dropna().astype(str)
    if vals.empty:
        return False

    interval_like = vals.str.contains(r"[\[\(].*,.*[\]\)]", regex=True).mean()
    keyword_like = vals.str.contains(r"bin|missing|special", case=False, regex=True).mean()
    numeric_ratio = pd.to_numeric(vals, errors="coerce").notna().mean()

    if interval_like >= 0.30 or keyword_like >= 0.30:
        return True

    # Integers from direct transforms can still be valid bin identifiers.
    unique_vals = vals.nunique()
    if numeric_ratio >= 0.95 and unique_vals <= COMMON_CONFIG["max_bins"] + 3:
        coerced = pd.to_numeric(vals, errors="coerce")
        if (coerced.dropna() % 1 == 0).all():
            return True

    return numeric_ratio < 0.80


def try_direct_riskbands_bin_transform(binner: Any, X: pd.DataFrame, feature: str) -> pd.Series | None:
    attempts = [
        lambda: binner.transform(X, metric="bins"),
        lambda: binner.transform(X, metric="bin"),
        lambda: binner.transform(X, output="bins"),
        lambda: binner.transform(X, output="bin"),
        lambda: binner.transform_bins(X),
        lambda: binner.bin_transform(X),
    ]

    for attempt in attempts:
        try:
            result = attempt()
            ser = coerce_series_from_transform(result, feature)
            if looks_like_bin_labels(ser):
                return ser.astype(str)
        except Exception:
            continue

    return None


def build_riskbands_bin_mapping_from_summary(
    transformed_series: pd.Series,
    feature_summary: pd.DataFrame,
) -> dict[Any, str]:
    if feature_summary.empty or "bin" not in feature_summary.columns:
        return {}

    working = feature_summary.copy()

    woe_col = None
    for candidate in ["woe", "mean_woe"]:
        if candidate in working.columns:
            woe_col = candidate
            break

    if woe_col is None:
        return {}

    working = working.dropna(subset=[woe_col]).copy()
    if working.empty:
        return {}

    working[woe_col] = pd.to_numeric(working[woe_col], errors="coerce")
    working = working.dropna(subset=[woe_col]).drop_duplicates(subset=[woe_col], keep="first")
    if working.empty:
        return {}

    mapping = {}
    for raw_val in pd.Series(transformed_series).dropna().unique():
        try:
            raw_float = float(raw_val)
        except Exception:
            continue

        direct_match = working.loc[np.isclose(working[woe_col], raw_float, atol=1e-8)]
        if not direct_match.empty:
            chosen = direct_match.iloc[0]
        else:
            distances = (working[woe_col] - raw_float).abs()
            chosen = working.loc[distances.idxmin()]

        mapping[rounded_key(raw_float)] = str(chosen["bin"])

    return mapping


def coerce_series_from_transform(result: Any, feature: str) -> pd.Series:
    if isinstance(result, pd.Series):
        return result.copy()
    if isinstance(result, pd.DataFrame):
        if feature in result.columns:
            return result[feature].copy()
        return result.iloc[:, 0].copy()
    return pd.Series(result, name=feature)


def extract_optuna_metadata(obj: Any) -> dict[str, Any]:
    meta = {}

    direct_attrs = [
        "best_params_",
        "best_params",
        "optuna_best_params_",
        "best_value_",
        "best_value",
        "optuna_best_value_",
    ]

    for attr in direct_attrs:
        if hasattr(obj, attr):
            value = getattr(obj, attr)
            if value is not None:
                meta[attr] = value

    study_attrs = ["study_", "study", "_study", "optuna_study_", "optuna_study"]
    for attr in study_attrs:
        if hasattr(obj, attr):
            study = getattr(obj, attr)
            try:
                if hasattr(study, "best_params"):
                    meta["study_best_params"] = study.best_params
                if hasattr(study, "best_value"):
                    meta["study_best_value"] = study.best_value
                if hasattr(study, "direction"):
                    meta["study_direction"] = str(study.direction)
            except Exception:
                pass

    for key, value in getattr(obj, "__dict__", {}).items():
        key_low = str(key).lower()
        if any(token in key_low for token in ["optuna", "best_param", "best_value"]):
            if key not in meta:
                if isinstance(value, (dict, list, str, int, float, type(None))):
                    meta[key] = value

    return meta


def prepare_eval_frames(
    *,
    train_df: pd.DataFrame,
    oot_df: pd.DataFrame,
    train_bins: pd.Series,
    oot_bins: pd.Series,
    train_woe: pd.Series,
    oot_woe: pd.Series,
    feature: str,
    target: str,
    time_col: str,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    train_eval = pd.DataFrame(
        {
            time_col: train_df[time_col].values,
            target: train_df[target].values,
            "bin": train_bins.astype(str).values,
            "woe": train_woe.astype(float).values,
            feature: train_df[feature].values,
            "sample": "train",
        }
    )

    oot_eval = pd.DataFrame(
        {
            time_col: oot_df[time_col].values,
            target: oot_df[target].values,
            "bin": oot_bins.astype(str).values,
            "woe": oot_woe.astype(float).values,
            feature: oot_df[feature].values,
            "sample": "oot",
        }
    )

    return train_eval, oot_eval


def evaluate_artifact(
    method_name: str,
    artifact: dict[str, Any],
    *,
    time_col: str,
    target_col: str,
) -> dict[str, Any]:
    train_eval = artifact["train_eval"].copy()
    oot_eval = artifact["oot_eval"].copy()

    train_eval["sample"] = "train"
    oot_eval["sample"] = "oot"
    full_eval = pd.concat([train_eval, oot_eval], ignore_index=True)

    train_summary = summarize_bins(train_eval, bin_col="bin", target_col=target_col, woe_col="woe")
    oot_summary = summarize_bins(oot_eval, bin_col="bin", target_col=target_col, woe_col="woe")

    ordered_bins = train_summary.sort_values("bad_rate")["bin"].astype(str).tolist()

    time_table = build_bin_time_table(
        train_eval,
        time_col=time_col,
        bin_col="bin",
        target_col=target_col
    )

    time_table_full = build_bin_time_table(
        full_eval,
        time_col=time_col,
        bin_col="bin",
        target_col=target_col
    )

    inversion_ratio = compute_inversion_ratio(
        time_table,
        ordered_bins=ordered_bins,
        time_col=time_col,
        bin_col="bin"
    )

    volatility = compute_event_rate_volatility(
        time_table,
        ordered_bins=ordered_bins,
        time_col=time_col,
        bin_col="bin"
    )

    psi_value, psi_table = psi_from_series(train_eval["bin"], oot_eval["bin"])

    train_auc, oot_auc = compute_univariate_auc(
        train_eval,
        oot_eval,
        woe_col="woe",
        target_col=target_col
    )

    sparse_bin_ratio = float(
        (
            train_summary["sparse_bin"].sum()
            + oot_summary["sparse_bin"].sum()
        )
        / max(train_summary.shape[0] + oot_summary.shape[0], 1)
    )

    temporal_score, temporal_penalty = temporal_score_from_metrics(
        inversion_ratio=inversion_ratio,
        event_rate_volatility=volatility,
        sparse_bin_ratio=sparse_bin_ratio,
        psi_value=psi_value,
        auc_drop=train_auc - oot_auc,
    )

    return {
        "method": method_name,
        "n_bins_train": int(train_summary.shape[0]),
        "train_auc": float(train_auc),
        "oot_auc": float(oot_auc),
        "auc_drop": float(train_auc - oot_auc),
        "psi_train_vs_oot": float(psi_value),
        "inversion_ratio": float(inversion_ratio),
        "event_rate_volatility": float(volatility),
        "sparse_bin_ratio": float(sparse_bin_ratio),
        "temporal_penalty": float(temporal_penalty),
        "temporal_score": float(temporal_score),
        "fit_time_sec": float(artifact["fit_time_sec"]),
        "train_summary": train_summary,
        "oot_summary": oot_summary,
        "time_table": time_table,
        "time_table_full": time_table_full,
        "psi_table": psi_table,
        "train_eval": train_eval,
        "oot_eval": oot_eval,
        "full_eval": full_eval,
        "optuna_metadata": artifact.get("optuna_metadata", {}),
    }


def fit_optimal_binning_pure(
    train_df: pd.DataFrame,
    oot_df: pd.DataFrame,
    *,
    feature: str,
    target: str,
    max_bins: int,
) -> dict[str, Any]:
    t0 = time.perf_counter()

    optb = OptimalBinning(
        name=feature,
        dtype="numerical",
        max_n_bins=max_bins,
    )
    optb.fit(train_df[feature].values, train_df[target].values)

    fit_time = time.perf_counter() - t0

    train_bins = pd.Series(
        optb.transform(train_df[feature].values, metric="bins"),
        index=train_df.index,
        name=feature
    )
    oot_bins = pd.Series(
        optb.transform(oot_df[feature].values, metric="bins"),
        index=oot_df.index,
        name=feature
    )

    train_woe = pd.Series(
        optb.transform(train_df[feature].values, metric="woe"),
        index=train_df.index,
        name=feature
    )
    oot_woe = pd.Series(
        optb.transform(oot_df[feature].values, metric="woe"),
        index=oot_df.index,
        name=feature
    )

    train_eval, oot_eval = prepare_eval_frames(
        train_df=train_df,
        oot_df=oot_df,
        train_bins=train_bins,
        oot_bins=oot_bins,
        train_woe=train_woe,
        oot_woe=oot_woe,
        feature=feature,
        target=target,
        time_col=TIME_COL,
    )

    return {
        "object": optb,
        "fit_time_sec": fit_time,
        "train_eval": train_eval,
        "oot_eval": oot_eval,
        "optuna_metadata": {},
    }


def fit_riskbands_variant(
    train_df: pd.DataFrame,
    oot_df: pd.DataFrame,
    *,
    feature: str,
    target: str,
    time_col: str,
    use_optuna: bool,
    max_bins: int,
    min_event_rate_diff: float,
    monotonic: str | None,
) -> dict[str, Any]:
    binner = Binner(
        strategy="supervised",
        max_bins=max_bins,
        min_event_rate_diff=min_event_rate_diff,
        monotonic=monotonic,
        check_stability=True,
        use_optuna=use_optuna,
        time_col=time_col,
        force_numeric=[feature],
    )

    X_train = train_df[[feature, time_col]].copy()
    y_train = train_df[target].copy()

    t0 = time.perf_counter()
    binner.fit(X_train, y_train, time_col=time_col)
    fit_time = time.perf_counter() - t0

    train_transformed = coerce_series_from_transform(binner.transform(train_df[[feature, time_col]]), feature)
    oot_transformed = coerce_series_from_transform(binner.transform(oot_df[[feature, time_col]]), feature)

    direct_train_bins = try_direct_riskbands_bin_transform(binner, train_df[[feature, time_col]], feature)
    direct_oot_bins = try_direct_riskbands_bin_transform(binner, oot_df[[feature, time_col]], feature)

    if direct_train_bins is not None and direct_oot_bins is not None:
        train_bins = direct_train_bins.astype(str)
        oot_bins = direct_oot_bins.astype(str)
    else:
        feature_summary = extract_riskbands_feature_summary(binner, feature)
        mapping = build_riskbands_bin_mapping_from_summary(train_transformed, feature_summary)

        if not mapping:
            mapping = build_riskbands_bin_mapping(train_transformed, feature_summary)

        train_bins = train_transformed.map(lambda x: mapping.get(rounded_key(x), f"unmapped_{rounded_key(x)}"))
        oot_bins = oot_transformed.map(lambda x: mapping.get(rounded_key(x), f"unmapped_{rounded_key(x)}"))

    train_eval, oot_eval = prepare_eval_frames(
        train_df=train_df,
        oot_df=oot_df,
        train_bins=train_bins,
        oot_bins=oot_bins,
        train_woe=train_transformed,
        oot_woe=oot_transformed,
        feature=feature,
        target=target,
        time_col=time_col,
    )

    return {
        "object": binner,
        "fit_time_sec": fit_time,
        "train_eval": train_eval,
        "oot_eval": oot_eval,
        "optuna_metadata": extract_optuna_metadata(binner) if use_optuna else {},
    }

## 7. Execução dos três cenários

In [7]:
artifacts = {}
errors = {}

run_plan = {
    "OptimalBinning puro": lambda: fit_optimal_binning_pure(
        train_df,
        oot_df,
        feature=FEATURE,
        target=TARGET,
        max_bins=COMMON_CONFIG["max_bins"],
    ),
    "RiskBands sem Optuna": lambda: fit_riskbands_variant(
        train_df,
        oot_df,
        feature=FEATURE,
        target=TARGET,
        time_col=TIME_COL,
        use_optuna=False,
        max_bins=COMMON_CONFIG["max_bins"],
        min_event_rate_diff=COMMON_CONFIG["min_event_rate_diff"],
        monotonic=COMMON_CONFIG["monotonic"],
    ),
    "RiskBands com Optuna": lambda: fit_riskbands_variant(
        train_df,
        oot_df,
        feature=FEATURE,
        target=TARGET,
        time_col=TIME_COL,
        use_optuna=True,
        max_bins=COMMON_CONFIG["max_bins"],
        min_event_rate_diff=COMMON_CONFIG["min_event_rate_diff"],
        monotonic=COMMON_CONFIG["monotonic"],
    ),
}

for method_name, runner in run_plan.items():
    try:
        artifacts[method_name] = runner()
        print(f"[OK] {method_name}")
    except Exception as e:
        errors[method_name] = repr(e)
        print(f"[ERRO] {method_name}: {repr(e)}")

if errors:
    print("\nErros encontrados:")
    print(json.dumps(errors, indent=2, ensure_ascii=False))

[OK] OptimalBinning puro
[OK] RiskBands sem Optuna

[I 2026-04-13 01:03:16,384] A new study created in memory with name: no-name-f225ed5f-2043-4843-b387-01ce62200f4a



[OK] RiskBands com Optuna


## 8. Tabela consolidada de métricas temporais

In [8]:
evaluations = []

for method_name, artifact in artifacts.items():
    evaluations.append(
        evaluate_artifact(
            method_name,
            artifact,
            time_col=TIME_COL,
            target_col=TARGET,
        )
    )

comparison_df = (
    pd.DataFrame(
        [
            {
                k: v
                for k, v in item.items()
                if not isinstance(v, (pd.DataFrame, dict))
            }
            for item in evaluations
        ]
    )
    .sort_values(["temporal_score", "oot_auc"], ascending=[False, False])
    .reset_index(drop=True)
)

display(comparison_df)

,method,n_bins_train,train_auc,oot_auc,auc_drop,psi_train_vs_oot,inversion_ratio,event_rate_volatility,sparse_bin_ratio,temporal_penalty,temporal_score,fit_time_sec
0,RiskBands com Optuna,5,0.589554,0.611402,-0.021848,1.323625,0.28125,0.054931,0.100000,43.670321,56.329679,2.316915
1,RiskBands sem Optuna,6,0.592308,0.611746,-0.019438,1.338001,0.35000,0.055754,0.083333,45.688602,54.311398,0.176667
2,OptimalBinning puro,6,0.590645,0.614274,-0.023629,1.372604,0.33750,0.060163,0.166667,47.665830,52.334170,0.070668


## 9. Leitura inicial: o que olhar primeiro

A tabela consolidada deve ser lida nesta ordem:

1. **`temporal_score`**: score sintético de qualidade temporal
2. **`oot_auc`**: robustez fora do período de treino
3. **`inversion_ratio`**: quanto a ordenação de risco se quebra mês a mês
4. **`psi_train_vs_oot`**: sensibilidade a drift de distribuição
5. **`sparse_bin_ratio`**: presença de bins pouco representativos
6. **`fit_time_sec`**: custo computacional

A tese do notebook é:

- o **OptimalBinning puro** pode performar bem estaticamente
- o **RiskBands** tende a se destacar quando o problema é temporal
- o **RiskBands com Optuna** deve ser avaliado pelo ganho incremental vs custo adicional


In [9]:
score_fig = make_subplots(
    rows=2,
    cols=3,
    subplot_titles=(
        "Score temporal consolidado",
        "AUC OOT",
        "Tempo de fit",
        "PSI treino vs OOT",
        "Inversion ratio",
        "Sparse bin ratio"
    )
)

metric_specs = [
    ("temporal_score", 1, 1),
    ("oot_auc", 1, 2),
    ("fit_time_sec", 1, 3),
    ("psi_train_vs_oot", 2, 1),
    ("inversion_ratio", 2, 2),
    ("sparse_bin_ratio", 2, 3),
]

for metric, row, col in metric_specs:
    score_fig.add_trace(
        go.Bar(
            x=comparison_df["method"],
            y=comparison_df[metric],
            name=metric,
            text=np.round(comparison_df[metric], 4),
            textposition="outside",
            showlegend=False,
        ),
        row=row,
        col=col,
    )

score_fig.update_layout(
    template="plotly_white",
    height=800,
    title="Placar comparativo entre as abordagens"
)

score_fig.show()

## 10. Comparação lado a lado dos bins

Aqui fica visível a diferença entre as partições produzidas por cada abordagem.

A leitura recomendada é:

- observar se existem bins muito estreitos ou muito frágeis
- comparar o `bad_rate` no treino e no OOT
- ver se a ordenação continua coerente fora da amostra


In [10]:
bin_table_frames = []

for item in evaluations:
    train_summary = item["train_summary"].copy()
    train_summary["method"] = item["method"]
    train_summary["sample"] = "train"

    oot_summary = item["oot_summary"].copy()
    oot_summary["method"] = item["method"]
    oot_summary["sample"] = "oot"

    bin_table_frames.append(train_summary)
    bin_table_frames.append(oot_summary)

bin_table_df = pd.concat(bin_table_frames, ignore_index=True)

display(bin_table_df.head(30))

,bin,count,bad_rate,mean_woe,share,sparse_bin,method,sample
0,"(-inf, 0.23)",2218,0.105050,0.393943,0.277250,False,OptimalBinning puro,train
1,"[0.23, 0.31)",1445,0.128028,0.170118,0.180625,False,OptimalBinning puro,train
2,"[0.31, 0.52)",2930,0.148805,-0.004392,0.366250,False,OptimalBinning puro,train
3,"[0.52, 0.56)",412,0.201456,-0.371176,0.051500,False,OptimalBinning puro,train
4,"[0.56, 0.62)",419,0.221957,-0.494095,0.052375,False,OptimalBinning puro,train
5,"[0.62, inf)",576,0.270833,-0.757994,0.072000,False,OptimalBinning puro,train
6,"[0.23, 0.31)",196,0.188776,0.170118,0.049000,True,OptimalBinning puro,oot
7,"(-inf, 0.23)",114,0.201754,0.393943,0.028500,True,OptimalBinning puro,oot
8,"[0.31, 0.52)",1276,0.221003,-0.004392,0.319000,False,OptimalBinning puro,oot
9,"[0.52, 0.56)",349,0.280802,-0.371176,0.087250,False,OptimalBinning puro,oot


In [11]:
methods_available = comparison_df["method"].tolist()

fig = make_subplots(
    rows=1,
    cols=len(methods_available),
    subplot_titles=methods_available
)

for idx, method_name in enumerate(methods_available, start=1):
    temp_train = (
        bin_table_df[
            (bin_table_df["method"] == method_name)
            & (bin_table_df["sample"] == "train")
        ]
        .copy()
    )
    temp_oot = (
        bin_table_df[
            (bin_table_df["method"] == method_name)
            & (bin_table_df["sample"] == "oot")
        ]
        .copy()
    )

    ordered_bins = temp_train.sort_values("bad_rate")["bin"].astype(str).tolist()

    temp_train["bin"] = pd.Categorical(temp_train["bin"].astype(str), categories=ordered_bins, ordered=True)
    temp_oot["bin"] = pd.Categorical(temp_oot["bin"].astype(str), categories=ordered_bins, ordered=True)

    temp_train = temp_train.sort_values("bin")
    temp_oot = temp_oot.sort_values("bin")

    fig.add_trace(
        go.Bar(
            x=temp_train["bin"].astype(str),
            y=temp_train["bad_rate"],
            name=f"{method_name} | train",
            legendgroup=f"{method_name}_train",
            showlegend=(idx == 1),
        ),
        row=1,
        col=idx,
    )

    fig.add_trace(
        go.Scatter(
            x=temp_oot["bin"].astype(str),
            y=temp_oot["bad_rate"],
            mode="lines+markers",
            name=f"{method_name} | oot",
            legendgroup=f"{method_name}_oot",
            showlegend=(idx == 1),
        ),
        row=1,
        col=idx,
    )

fig.update_layout(
    template="plotly_white",
    height=520,
    title=f"Comparação lado a lado dos bins para {FEATURE}"
)

fig.update_yaxes(title_text="Bad rate", row=1, col=1)
fig.show()

## 11. Heatmap de estabilidade temporal

O gráfico abaixo mostra a evolução do `bad_rate` por safra e por bin, agora usando **treino + OOT** para evidenciar o comportamento completo ao longo do tempo.

Um bom binning temporal tende a apresentar:

- menos trocas de ordem entre bins
- menos oscilações abruptas mês a mês
- leitura mais consistente mesmo com drift


In [19]:
subplot_titles = []
heatmap_payload = []

for item in evaluations:
    time_table = item["time_table"].copy()
    ordered_bins = item["train_summary"].sort_values("bad_rate")["bin"].astype(str).tolist()

    pivot = (
        time_table.assign(bin=time_table["bin"].astype(str))
        .pivot(index=TIME_COL, columns="bin", values="bad_rate")
        .reindex(columns=ordered_bins)
        .sort_index()
    )

    values = pivot.values.astype(float)

    # 1) Instabilidade temporal:
    # média das diferenças absolutas entre safras consecutivas, por bin
    temporal_instability = np.nanmean(np.abs(np.diff(values, axis=0)))

    # 2) Penalização de inversão:
    # conta quanto os bins deixam de respeitar a ordem crescente de risco em cada safra
    monotonicity_penalty = 0.0
    for row in values:
        diffs = np.diff(row)
        negative_diffs = diffs[diffs < 0]
        if len(negative_diffs) > 0:
            monotonicity_penalty += np.abs(negative_diffs).sum()

    monotonicity_penalty = monotonicity_penalty / values.shape[0]

    # Score final: menor = melhor
    stability_score = temporal_instability + monotonicity_penalty

    subplot_titles.append(
        f"{item['method']}<br><sup>score={stability_score:.4f} | "
        f"temp={temporal_instability:.4f} | mono={monotonicity_penalty:.4f}</sup>"
    )

    heatmap_payload.append(
        {
            "item": item,
            "pivot": pivot,
            "score": stability_score,
            "temporal_instability": temporal_instability,
            "monotonicity_penalty": monotonicity_penalty,
        }
    )

heatmap_payload = sorted(heatmap_payload, key=lambda x: x["score"])
subplot_titles = [
    f"{x['item']['method']}<br><sup>score={x['score']:.4f} | "
    f"temp={x['temporal_instability']:.4f} | mono={x['monotonicity_penalty']:.4f}</sup>"
    for x in heatmap_payload
]

heatmap_fig = make_subplots(
    rows=1,
    cols=len(heatmap_payload),
    subplot_titles=subplot_titles
)

for idx, payload in enumerate(heatmap_payload, start=1):
    pivot = payload["pivot"]

    heatmap_fig.add_trace(
        go.Heatmap(
            z=pivot.values,
            x=[str(c) for c in pivot.columns],
            y=[str(i) for i in pivot.index],
            text=np.round(pivot.values, 3),
            texttemplate="%{text}",
            textfont={"size": 9},
            colorbar={"title": "bad_rate"} if idx == len(heatmap_payload) else None,
            showscale=(idx == len(heatmap_payload)),
        ),
        row=1,
        col=idx,
    )

heatmap_fig.update_layout(
    template="plotly_white",
    height=560,
    title=f"Heatmap de estabilidade temporal - {FEATURE}<br><sup>Menor score = melhor estabilidade</sup>"
)

heatmap_fig.show()

## 📖 Como interpretar o score de estabilidade do heatmap

Para facilitar a comparação entre os métodos, cada heatmap exibe um **score resumo de estabilidade temporal**.

A ideia é transformar a leitura visual do heatmap em uma **métrica quantitativa simples e interpretável**.

---

### 🎯 O que esse score resume

O score é composto por dois blocos principais:

### **temp**
Representa o quanto o binning está **“nervoso” ao longo do tempo**.

Na prática, mede a **variação média do `bad_rate` entre safras consecutivas**, dentro de cada bin.

- valor baixo → transições suaves
- valor alto → oscilações bruscas
- ajuda a detectar serrilhado temporal

👉 Quanto menor, melhor a estabilidade.

---

### **mono**
Representa a **violação da ordem esperada entre bins**.

Como os bins são organizados do menor para o maior risco no treino, espera-se que essa ordem seja preservada nas demais safras.

Esse componente penaliza situações onde:

- bins à esquerda ficam mais arriscados que bins à direita
- ocorre inversão de monotonicidade
- a separação de risco se deteriora no tempo

👉 Quanto menor, melhor a consistência do ranking de risco.

---

### **score**
É a soma dos dois componentes:

```text
score = temp + mono
```

Esse valor resume o equilíbrio entre:

- suavidade temporal
- preservação da ordem de risco

👉 **menor score = melhor heatmap**

---

# 🧠 Como a leitura deve ser feita

Ao comparar os subplots:

## ✅ Melhor método
Procure o método com:

- **menor `score`**
- menor `temp`
- menor `mono`
- gradiente de cores mais suave
- menos inversões entre bins
- menos efeito “zebra” entre safras

Esse será o setup com maior potencial de robustez temporal.

---

## 📌 Interpretação prática

### Menor `score`
➡️ melhor estabilidade geral

### Menor `temp`
➡️ bins mais suaves no tempo

### Menor `mono`
➡️ bins mais monotônicos e organizados

---

## 🏆 Regra prática
Em cenários de risco de crédito, o método vencedor tende a ser aquele que mantém:

- boa separação entre bins
- baixa volatilidade entre safras
- ranking de risco persistente

Esse score ajuda a transformar essa análise em uma **comparação objetiva entre setups**, incluindo:

- RiskBands padrão
- RiskBands com Optuna
- OptimalBinning puro

## 12. Curvas de event rate por vintage e por bin

Este painel agora usa **todas as safras (treino + OOT)** e colore os bins por **rank de risco dentro de cada método**, evitando a sensação de que o subplot do RiskBands está “herdando” a legenda e as cores do primeiro gráfico.

Ele responde diretamente à pergunta:

- os bins continuam separados ao longo das safras?
- a hierarquia de risco permanece consistente?
- há cruzamentos frequentes entre curvas?

Quanto mais cruzamentos, pior a estabilidade temporal.


In [13]:
rank_palette = [
    "#1f77b4",
    "#ff7f0e",
    "#2ca02c",
    "#d62728",
    "#9467bd",
    "#8c564b",
    "#e377c2",
    "#7f7f7f",
]

curve_fig = make_subplots(
    rows=len(methods_available),
    cols=1,
    subplot_titles=methods_available,
    shared_xaxes=True,
    vertical_spacing=0.08
)

oot_start = oot_df[TIME_COL].min()

for idx, item in enumerate(evaluations, start=1):
    time_table = item.get("time_table_full", item["time_table"]).copy()

    ordered_bins = (
        item["train_summary"]
        .sort_values("bad_rate")["bin"]
        .astype(str)
        .tolist()
    )

    time_table["bin"] = time_table["bin"].astype(str)
    time_table = time_table.loc[time_table["bin"].isin(ordered_bins)].copy()
    time_table = time_table.sort_values([TIME_COL, "bin"])

    for rank_idx, bin_label in enumerate(ordered_bins, start=1):
        temp = time_table.loc[time_table["bin"] == str(bin_label)].copy()
        temp = temp.sort_values(TIME_COL)

        if temp.empty:
            continue

        customdata = np.column_stack([
            np.repeat(item["method"], len(temp)),
            np.repeat(bin_label, len(temp)),
            np.repeat(rank_idx, len(temp)),
            temp["count"].to_numpy(),
        ])

        curve_fig.add_trace(
            go.Scatter(
                x=temp[TIME_COL],
                y=temp["bad_rate"],
                mode="lines+markers",
                name=f"Rank {rank_idx} | {bin_label}",
                legendgroup=f"rank_{rank_idx}",
                line={"color": rank_palette[(rank_idx - 1) % len(rank_palette)], "width": 2},
                marker={"size": 7},
                customdata=customdata,
                hovertemplate=(
                    "Método=%{customdata[0]}<br>"
                    "Bin=%{customdata[1]}<br>"
                    "Rank=%{customdata[2]}<br>"
                    "Safra=%{x}<br>"
                    "Bad rate=%{y:.4f}<br>"
                    "Count=%{customdata[3]}<extra></extra>"
                ),
                showlegend=(idx == 1),
            ),
            row=idx,
            col=1,
        )

    curve_fig.update_yaxes(title_text="Bad rate", row=idx, col=1)

    if pd.notna(oot_start):
        curve_fig.add_vline(
            x=oot_start,
            line_width=1,
            line_dash="dash",
            line_color="gray",
            row=idx,
            col=1,
        )

curve_fig.update_layout(
    template="plotly_white",
    height=420 * len(methods_available),
    title=f"Curvas de event rate por vintage (treino + OOT) - {FEATURE}",
    legend_title="Rank do bin no treino",
    hovermode="x unified"
)

curve_fig.show()

## 13. Evolução do WoE médio por safra

Mesmo sendo uma simplificação, o WoE médio por safra ajuda a visualizar:

- quanto a representação da variável oscila ao longo do tempo
- se a mudança de regime gera salto abrupto
- se a abordagem produz uma leitura temporal mais regular

In [14]:
woe_fig = go.Figure()

for item in evaluations:
    full_eval = item.get("full_eval")

    if full_eval is None:
        artifact = artifacts.get(item["method"], {})
        train_eval = artifact.get("train_eval")
        oot_eval = artifact.get("oot_eval")

        if train_eval is None or oot_eval is None:
            raise KeyError(
                f"Não encontrei 'full_eval' nem 'train_eval'/'oot_eval' para o método {item['method']}."
            )

        full_eval = pd.concat([train_eval, oot_eval], ignore_index=True)

    woe_month = (
        full_eval.groupby(TIME_COL)["woe"]
        .mean()
        .reset_index()
        .sort_values(TIME_COL)
    )

    woe_fig.add_trace(
        go.Scatter(
            x=woe_month[TIME_COL],
            y=woe_month["woe"],
            mode="lines+markers",
            name=item["method"]
        )
    )

if pd.notna(oot_df[TIME_COL].min()):
    woe_fig.add_vline(
        x=oot_df[TIME_COL].min(),
        line_width=1,
        line_dash="dash",
        line_color="gray",
    )

woe_fig.update_layout(
    template="plotly_white",
    height=420,
    title=f"Evolução do WoE médio por safra (treino + OOT) - {FEATURE}",
    yaxis_title="WoE médio"
)

woe_fig.show()

## 14. Custo-benefício: ganho temporal vs tempo de processamento

Este gráfico ajuda a responder:

- o ganho do RiskBands é material?
- a busca com Optuna trouxe melhora relevante?
- o tempo adicional compensou?

In [15]:
cost_benefit_fig = go.Figure()

cost_benefit_fig.add_trace(
    go.Scatter(
        x=comparison_df["fit_time_sec"],
        y=comparison_df["temporal_score"],
        mode="markers+text",
        text=comparison_df["method"],
        textposition="top center",
        name="métodos"
    )
)

cost_benefit_fig.update_layout(
    template="plotly_white",
    height=450,
    title="Custo-benefício: tempo de fit vs score temporal",
    xaxis_title="Tempo de fit (segundos)",
    yaxis_title="Score temporal consolidado"
)

cost_benefit_fig.show()

## 15. Parâmetros otimizados pelo Optuna, se disponíveis

In [16]:
optuna_eval = next((item for item in evaluations if item["method"] == "RiskBands com Optuna"), None)

if optuna_eval is None:
    print("Abordagem com Optuna não foi executada.")
else:
    optuna_meta = optuna_eval.get("optuna_metadata", {})
    if not optuna_meta:
        print(
            "A execução com Optuna ocorreu, mas nenhum atributo padronizado de melhor parâmetro "
            "foi encontrado automaticamente nesta versão do objeto."
        )
    else:
        optuna_meta_df = pd.DataFrame(
            [{"attribute": k, "value": json.dumps(v, ensure_ascii=False) if isinstance(v, (dict, list)) else v}
             for k, v in optuna_meta.items()]
        )
        display(optuna_meta_df)

,attribute,value
0,best_params_,"{""utilizacao"": {""max_bins"": 5, ""min_bin_size"":..."
1,use_optuna,True


## 16. Ganho incremental do Optuna vs RiskBands sem Optuna

In [17]:
rb_no = comparison_df.loc[comparison_df["method"].eq("RiskBands sem Optuna")]
rb_opt = comparison_df.loc[comparison_df["method"].eq("RiskBands com Optuna")]

if rb_no.empty or rb_opt.empty:
    print("Não foi possível calcular o ganho incremental porque uma das versões do RiskBands não rodou.")
else:
    delta_df = pd.DataFrame(
        {
            "métrica": [
                "temporal_score",
                "oot_auc",
                "psi_train_vs_oot",
                "inversion_ratio",
                "sparse_bin_ratio",
                "fit_time_sec",
            ],
            "delta_optuna_menos_sem_optuna": [
                float(rb_opt["temporal_score"].iloc[0] - rb_no["temporal_score"].iloc[0]),
                float(rb_opt["oot_auc"].iloc[0] - rb_no["oot_auc"].iloc[0]),
                float(rb_opt["psi_train_vs_oot"].iloc[0] - rb_no["psi_train_vs_oot"].iloc[0]),
                float(rb_opt["inversion_ratio"].iloc[0] - rb_no["inversion_ratio"].iloc[0]),
                float(rb_opt["sparse_bin_ratio"].iloc[0] - rb_no["sparse_bin_ratio"].iloc[0]),
                float(rb_opt["fit_time_sec"].iloc[0] - rb_no["fit_time_sec"].iloc[0]),
            ],
        }
    )
    display(delta_df)

    fig = go.Figure(
        data=[
            go.Bar(
                x=delta_df["métrica"],
                y=delta_df["delta_optuna_menos_sem_optuna"],
                text=np.round(delta_df["delta_optuna_menos_sem_optuna"], 6),
                textposition="outside",
            )
        ]
    )

    fig.update_layout(
        template="plotly_white",
        height=420,
        title="Ganho incremental do Optuna sobre o RiskBands sem Optuna"
    )

    fig.show()

,métrica,delta_optuna_menos_sem_optuna
0,temporal_score,2.018282
1,oot_auc,-0.000344
2,psi_train_vs_oot,-0.014376
3,inversion_ratio,-0.068750
4,sparse_bin_ratio,0.016667
5,fit_time_sec,2.140248


## 17. Conclusão interpretativa automática

A célula abaixo gera uma leitura executiva com base nos resultados efetivamente obtidos na execução.


In [18]:
best_temporal = comparison_df.sort_values("temporal_score", ascending=False).iloc[0]
best_oot = comparison_df.sort_values("oot_auc", ascending=False).iloc[0]
fastest = comparison_df.sort_values("fit_time_sec", ascending=True).iloc[0]

print("=== RESUMO EXECUTIVO ===\n")

print(f"Melhor score temporal: {best_temporal['method']} "
      f"(score={best_temporal['temporal_score']:.2f})")

print(f"Melhor AUC OOT: {best_oot['method']} "
      f"(oot_auc={best_oot['oot_auc']:.4f})")

print(f"Menor custo computacional: {fastest['method']} "
      f"(fit_time_sec={fastest['fit_time_sec']:.4f})")

print("\n=== QUANDO USAR CADA ABORDAGEM ===\n")

print("1. OptimalBinning puro")
print("- Use quando o objetivo principal for rapidez, baseline forte e problema relativamente estável no tempo.")
print("- Faz mais sentido quando drift temporal e mudança de regime não são o principal risco do projeto.")

print("\n2. RiskBands sem Optuna")
print("- Use quando a estabilidade temporal for parte central da decisão.")
print("- É a opção natural para cenários em que bins instáveis, inversões entre safras e robustez OOT importam mais do que um pequeno ganho estático.")

print("\n3. RiskBands com Optuna")
print("- Use quando houver margem de tempo computacional e o ganho incremental em score temporal ou OOT AUC for material.")
print("- Em geral, vale mais a pena em bases mais sensíveis, com maior drift, ou quando a variável é realmente crítica para o score final.")

rb_no = comparison_df.loc[comparison_df["method"].eq("RiskBands sem Optuna")]
rb_opt = comparison_df.loc[comparison_df["method"].eq("RiskBands com Optuna")]

if not rb_no.empty and not rb_opt.empty:
    delta_score = float(rb_opt["temporal_score"].iloc[0] - rb_no["temporal_score"].iloc[0])
    delta_auc = float(rb_opt["oot_auc"].iloc[0] - rb_no["oot_auc"].iloc[0])
    delta_time = float(rb_opt["fit_time_sec"].iloc[0] - rb_no["fit_time_sec"].iloc[0])

    print("\n=== LEITURA ESPECÍFICA SOBRE OPTUNA ===\n")
    print(f"Ganho em temporal_score: {delta_score:.4f}")
    print(f"Ganho em OOT AUC: {delta_auc:.6f}")
    print(f"Custo adicional de tempo: {delta_time:.4f} segundos")

    optuna_worth_it = (delta_score >= 2.0) or (delta_auc >= 0.005)

    if optuna_worth_it:
        print("Conclusão: nesta execução, habilitar Optuna parece justificável.")
    else:
        print("Conclusão: nesta execução, o ganho incremental do Optuna parece pequeno frente ao custo adicional.")

print("\n=== RECOMENDAÇÃO DE USO EM RISCO DE CRÉDITO ===\n")
print("- PD / application score: priorize estabilidade temporal e robustez OOT; RiskBands tende a fazer mais sentido.")
print("- Behavior score: drift de carteira e mudança de regime costumam pesar bastante; RiskBands geralmente ganha relevância.")
print("- LGD: quando a variável binada será usada mais como suporte interpretativo e menos como peça central de estabilidade temporal, o benchmark puro pode seguir suficiente em alguns casos.")

=== RESUMO EXECUTIVO ===

Melhor score temporal: RiskBands com Optuna (score=56.33)
Melhor AUC OOT: OptimalBinning puro (oot_auc=0.6143)
Menor custo computacional: OptimalBinning puro (fit_time_sec=0.0707)

=== QUANDO USAR CADA ABORDAGEM ===

1. OptimalBinning puro
- Use quando o objetivo principal for rapidez, baseline forte e problema relativamente estável no tempo.
- Faz mais sentido quando drift temporal e mudança de regime não são o principal risco do projeto.

2. RiskBands sem Optuna
- Use quando a estabilidade temporal for parte central da decisão.
- É a opção natural para cenários em que bins instáveis, inversões entre safras e robustez OOT importam mais do que um pequeno ganho estático.

3. RiskBands com Optuna
- Use quando houver margem de tempo computacional e o ganho incremental em score temporal ou OOT AUC for material.
- Em geral, vale mais a pena em bases mais sensíveis, com maior drift, ou quando a variável é realmente crítica para o score final.

=== LEITURA ESPECÍFICA

## 18. Mensagem final

Se o notebook mostrar que o **OptimalBinning puro** tem bom desempenho estático, mas perde em:

- `inversion_ratio`
- `psi_train_vs_oot`
- `sparse_bin_ratio`
- `temporal_score`

então a conclusão prática fica clara:

> **o valor do RiskBands não está apenas em “separar bem”, mas em separar de forma mais confiável ao longo do tempo.**

Esse é exatamente o ponto que costuma importar em produção para modelos de risco.
